# NumericalSystemModel

Goal: attach the numerics (1/h regularisation, reconstruction, Riemann solver) to a `SystemModel`.

Prerequisite: [Model and SystemModel](model_system_model.ipynb).

In [1]:
from zoomy_core.model.models import SME, Newtonian, NavierSlip, StressFree
from zoomy_core.numerics import NumericalSystemModel, ReconstructionSpec
from zoomy_core.systemmodel import SystemModel

sm = SystemModel.from_model(SME(level=0, closures=[Newtonian(), NavierSlip(), StressFree()]))
nsm = NumericalSystemModel.from_system_model(
    sm, reconstruction=ReconstructionSpec(order=1), depth_regularization="aux")
nsm.describe()

**SystemModel** — 3 equations, 1 spatial dimension

**Parameters:** $g = 9.81$, $\rho = 1.0$, $\nu = 0.0$, $\lambda_{s} = 0.0$, $e_{x} = 0.0$

**System form:**

$$M(Q)\,\partial_t Q + \nabla\cdot\!\big(F(Q) + P(Q)\big) + \sum_{d} B_{d}(Q)\,\partial_{d} Q - \nabla\cdot\!\big(A(Q)\,\nabla Q\big) - S(Q) = 0$$

**State $Q$:**

$$
\left[\begin{matrix}b\\h\\q_{0}\end{matrix}\right]
$$

**Auxiliary $Q_{aux}$:**

$$
\left[\begin{matrix}dq0dx\\dhdx\\dbdx\\hinv\end{matrix}\right]
$$

**Mass matrix $M$:**

$$
\left[\begin{matrix}1 & 0 & 0\\0 & 1 & 0\\0 & 0 & 1\end{matrix}\right]
$$

**Flux $F$:**

$$
\left[\begin{matrix}0\\q_{0}\\hinv q_{0}^{2}\end{matrix}\right]
$$

**Hydrostatic pressure $P$:**

$$
\left[\begin{matrix}0\\0\\\frac{g h^{2}}{2}\end{matrix}\right]
$$

**Diffusive flux $A$ $= 0$**

**NCP $B$:**

$$
\left[\begin{matrix}0 & 0 & 0\\0 & 0 & 0\\g h & 0 & 0\end{matrix}\right]
$$

**Source $S$ (implicit):**

$$
\left[\begin{matrix}0\\0\\e_{x} g h - \frac{hinv \lambda_{s} q_{0}}{\rho}\end{matrix}\right]
$$

**Operations:** from_model, declare_roles, declare_tags, declare_roles, declare_tags, normalize_face_normal, register_aux, regularize_depth_aux, gate_eigenvalues_dry

In [2]:
print("SystemModel flux          ", sm.flux.tolist())
print("NumericalSystemModel flux ", nsm.flux.tolist())
print("aux update                ", nsm.update_aux_variables.tolist())
print("Riemann solver            ", nsm.riemann.__name__)
print("reconstruction            ", nsm.reconstruction.order, nsm.reconstruction.limiter)

SystemModel flux           [[0], [q_0], [q_0**2/h]]
NumericalSystemModel flux  [[0], [q_0], [hinv*q_0**2]]
aux update                 [[dq0dx], [dhdx], [dbdx], [sqrt(2)*h/sqrt(h**4 + Max(0.01, h)**4)]]
Riemann solver             PositiveNonconservativeRusanov
reconstruction             1 venkatakrishnan


In [3]:
assert [str(a) for a in nsm.aux_state][-1] == "hinv"
assert nsm.riemann.__name__ == "PositiveNonconservativeRusanov"
assert nsm.reconstruction.order == 1
assert "hinv" not in [str(a) for a in sm.aux_state]